In [4]:
import pandas as pd

customers = pd.read_csv('../data/olist_customers_dataset.csv')
orders = pd.read_csv('../data/olist_orders_dataset.csv')
payments = pd.read_csv('../data/olist_order_payments_dataset.csv')
items = pd.read_csv('../data/olist_order_items_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
sellers = pd.read_csv('../data/olist_sellers_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
category = pd.read_csv('../data/product_category_name_translation.csv')

print("all loaded")

all loaded


In [5]:
# CLEANING CHECKLIST (from Day 2 exploration)
# -----------------------------------------------
# 1. Date columns as strings       → convert to datetime
# 2. 610 null product categories   → fill with "unknown"
# 3. Column name typo in products  → rename "lenght" to "length"
# 4. Non-delivered orders          → filter, keep only delivered
# 5. Review comment columns        → drop (mostly empty, not needed)

In [6]:
# convert date columns from string to datetime
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print(orders.dtypes)

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [7]:
# keep only delivered orders
orders = orders[orders['order_status'] == 'delivered']
print("orders shape after filter:", orders.shape)

orders shape after filter: (96478, 8)


In [8]:
# fix typo in column names
products = products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

# fill null categories with "unknown"
products['product_category_name'] = products['product_category_name'].fillna('unknown')

print("null categories remaining:", products['product_category_name'].isnull().sum())
print(products.columns.tolist())

null categories remaining: 0
['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [9]:
# drop comment columns from reviews - mostly empty, not needed
reviews = reviews.drop(columns=['review_comment_title', 'review_comment_message'])
print(reviews.columns.tolist())

['review_id', 'order_id', 'review_score', 'review_creation_date', 'review_answer_timestamp']


In [10]:
# drop rows where delivery date is null (order not actually delivered)
orders = orders.dropna(subset=['order_delivered_customer_date'])
print("orders shape after dropping null delivery dates:", orders.shape)

orders shape after dropping null delivery dates: (96470, 8)


In [11]:
# step 1 — merge orders + customers
df = orders.merge(customers, on='customer_id', how='left')
print("after orders + customers:", df.shape)

after orders + customers: (96470, 12)


In [13]:
# step 2 — merge with order items
df = df.merge(items, on='order_id', how='left')
print("after + items:", df.shape)

# step 3 — merge with payments
df = df.merge(payments, on='order_id', how='left')
print("after + payments:", df.shape)

# step 4 — merge with products
df = df.merge(products, on='product_id', how='left')
print("after + products:", df.shape)

# step 5 — merge with category translation
df = df.merge(category, on='product_category_name', how='left')
print("after + category:", df.shape)

after + items: (110189, 18)
after + payments: (115030, 22)
after + products: (115030, 30)
after + category: (115030, 31)


In [14]:
# save final master dataframe
df.to_csv('../data/cleaned_master.csv', index=False)
print("saved. shape:", df.shape)
print("columns:", df.columns.tolist())

saved. shape: (115030, 31)
columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english']
